In [106]:
import pickle
import json

with open("../data/results/transcript_by_sentence.json", "r") as f:
    sentence_transcript_data = json.load(f)


with open("../data/results/Youtube_transcription.pkl", "rb") as f:
    data = pickle.load(f)

In [107]:
data

{'EWvNQjAaOHw': {'transcription': FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hi everyone so in this video I would', start=0.12, duration=3.72), FetchedTranscriptSnippet(text='like to continue our general audience', start=2.159, duration=5.041), FetchedTranscriptSnippet(text='series on large language models like', start=3.84, duration=5.839), FetchedTranscriptSnippet(text='chpd now in the previous video deep dive', start=7.2, duration=3.88), FetchedTranscriptSnippet(text='into llms that you can find on my', start=9.679, duration=3.281), FetchedTranscriptSnippet(text='YouTube we went into a lot of the', start=11.08, duration=3.2), FetchedTranscriptSnippet(text='underhood fundamentals of how these', start=12.96, duration=3.2), FetchedTranscriptSnippet(text='models are trained and how you should', start=14.28, duration=3.8), FetchedTranscriptSnippet(text='think about their cognition or', start=16.16, duration=4.84), FetchedTranscriptSnippet(text='psychology now in this vide

In [122]:
bool({1})

True

In [2]:
from urllib.parse import urlparse, parse_qs

def extract_video_id(url: str) -> str | None:
    """
    YouTube URL에서 videoId를 추출.
    지원 예시:
     - https://www.youtube.com/watch?v=VIDEO_ID
     - https://youtu.be/VIDEO_ID
     - https://www.youtube.com/embed/VIDEO_ID
     - https://www.youtube.com/v/VIDEO_ID
    """
    parsed = urlparse(url)
    host = parsed.netloc
    path = parsed.path

    # 예: youtu.be/VIDEO_ID
    if host in ("youtu.be", "www.youtu.be"):
        # path 시작 "/" 이후 부분이 ID
        return path.lstrip("/")

    # 예: www.youtube.com/watch?v=VIDEO_ID
    if host in ("www.youtube.com", "youtube.com", "m.youtube.com"):
        # 쿼리 파라미터 중 v 값을 봄
        qs = parse_qs(parsed.query)
        if "v" in qs:
            return qs["v"][0]

        # embed 형식 등
        # /embed/VIDEO_ID
        if path.startswith("/embed/"):
            return path.split("/")[2]
        # /v/VIDEO_ID
        if path.startswith("/v/"):
            return path.split("/")[2]

    return None  # ID를 못 찾았을 경우

In [13]:
import os
import json
import yaml
import requests
import pandas as pd
import numpy as np
from googleapiclient.discovery import build
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=API_KEY)
video_id_list = []

with open("../data/external_data/video_link_target.yaml", "r") as f:
    video_url_list = yaml.safe_load(f)

for video_url in video_url_list['target_video_link']:
    video_id = extract_video_id(video_url)
    video_id_list.append(video_id)

request = youtube.videos().list(
    part = "snippet",
    regionCode = "US",
    id = video_id_list
)
response = request.execute()

In [53]:
video_url_list

{'target_video_link': ['https://www.youtube.com/watch?v=EWvNQjAaOHw',
  'https://www.youtube.com/watch?v=7xTGNNLPyMI']}

In [52]:
response['items'][0]

{'kind': 'youtube#video',
 'etag': 'H0ENrgCzNr0mIEsPsHKuuDbLhDY',
 'id': 'EWvNQjAaOHw',
 'snippet': {'publishedAt': '2025-02-27T22:29:56Z',
  'channelId': 'UCXUPKJO5MZQN11PqgIvyuvQ',
  'title': 'How I use LLMs',
  'description': "The example-driven, practical walkthrough of Large Language Models and their growing list of related features, as a new entry to my general audience series on LLMs. In this more practical followup, I take you through the many ways I use LLMs in my own life.\n\nChapters\n00:00:00 Intro into the growing LLM ecosystem\n00:02:54 ChatGPT interaction under the hood\n00:13:12 Basic LLM interactions examples\n00:18:03 Be aware of the model you're using, pricing tiers\n00:22:54 Thinking models and when to use them\n00:31:00 Tool use: internet search\n00:42:04 Tool use: deep research\n00:50:57 File uploads, adding documents to context\n00:59:00 Tool use: python interpreter, messiness of the ecosystem\n01:04:35 ChatGPT Advanced Data Analysis, figures, plots\n01:09:00 Cla

In [32]:
response['items'][0]['snippet'].keys()

dict_keys(['publishedAt', 'channelId', 'title', 'description', 'thumbnails', 'channelTitle', 'categoryId', 'liveBroadcastContent', 'defaultLanguage', 'localized', 'defaultAudioLanguage'])

In [34]:
response['items'][0]['snippet']['publishedAt']

'2025-02-27T22:29:56Z'

In [35]:
response['items'][0]['snippet']['channelId']

'UCXUPKJO5MZQN11PqgIvyuvQ'

In [36]:
response['items'][0]['snippet']['title'] # need to save in metadata

'How I use LLMs'

In [37]:
response['items'][0]['snippet']['description'] # can be used on fixing auto transcript with chatgpt

"The example-driven, practical walkthrough of Large Language Models and their growing list of related features, as a new entry to my general audience series on LLMs. In this more practical followup, I take you through the many ways I use LLMs in my own life.\n\nChapters\n00:00:00 Intro into the growing LLM ecosystem\n00:02:54 ChatGPT interaction under the hood\n00:13:12 Basic LLM interactions examples\n00:18:03 Be aware of the model you're using, pricing tiers\n00:22:54 Thinking models and when to use them\n00:31:00 Tool use: internet search\n00:42:04 Tool use: deep research\n00:50:57 File uploads, adding documents to context\n00:59:00 Tool use: python interpreter, messiness of the ecosystem\n01:04:35 ChatGPT Advanced Data Analysis, figures, plots\n01:09:00 Claude Artifacts, apps, diagrams\n01:14:02 Cursor: Composer, writing code\n01:22:28 Audio (Speech) Input/Output\n01:27:37 Advanced Voice Mode aka true audio inside the model\n01:37:09 NotebookLM, podcast generation\n01:40:20 Image i

In [39]:
response['items'][0]['snippet']['thumbnails']

{'default': {'url': 'https://i.ytimg.com/vi/EWvNQjAaOHw/default.jpg',
  'width': 120,
  'height': 90},
 'medium': {'url': 'https://i.ytimg.com/vi/EWvNQjAaOHw/mqdefault.jpg',
  'width': 320,
  'height': 180},
 'high': {'url': 'https://i.ytimg.com/vi/EWvNQjAaOHw/hqdefault.jpg',
  'width': 480,
  'height': 360},
 'standard': {'url': 'https://i.ytimg.com/vi/EWvNQjAaOHw/sddefault.jpg',
  'width': 640,
  'height': 480},
 'maxres': {'url': 'https://i.ytimg.com/vi/EWvNQjAaOHw/maxresdefault.jpg',
  'width': 1280,
  'height': 720}}

channelTitle', 'categoryId', 'liveBroadcastContent', 'defaultLanguage', 'localized', 'defaultAudioLanguage

In [41]:
response['items'][0]['snippet']['channelTitle'] # need to save in metadata

'Andrej Karpathy'

In [43]:
response['items'][0]['snippet']['categoryId'] # I don't know

'28'

In [44]:
response['items'][0]['snippet']['liveBroadcastContent'] # I don't know

'none'

In [46]:
response['items'][0]['snippet']['defaultLanguage'] # need to save in metadata

'en'

In [47]:
response['items'][0]['snippet']['localized'] # I don't know

{'title': 'How I use LLMs',
 'description': "The example-driven, practical walkthrough of Large Language Models and their growing list of related features, as a new entry to my general audience series on LLMs. In this more practical followup, I take you through the many ways I use LLMs in my own life.\n\nChapters\n00:00:00 Intro into the growing LLM ecosystem\n00:02:54 ChatGPT interaction under the hood\n00:13:12 Basic LLM interactions examples\n00:18:03 Be aware of the model you're using, pricing tiers\n00:22:54 Thinking models and when to use them\n00:31:00 Tool use: internet search\n00:42:04 Tool use: deep research\n00:50:57 File uploads, adding documents to context\n00:59:00 Tool use: python interpreter, messiness of the ecosystem\n01:04:35 ChatGPT Advanced Data Analysis, figures, plots\n01:09:00 Claude Artifacts, apps, diagrams\n01:14:02 Cursor: Composer, writing code\n01:22:28 Audio (Speech) Input/Output\n01:27:37 Advanced Voice Mode aka true audio inside the model\n01:37:09 Note

In [48]:
response['items'][0]['snippet']['defaultAudioLanguage'] # I don't know

'en'

In [56]:
import os
import json
import yaml
import requests
import pandas as pd
import numpy as np
from googleapiclient.discovery import build
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=API_KEY)
video_id_list = []

# === Function definition ===
def input_file_loader(input_file_path:str="../data/external_data/video_link_target.yaml"):
    with open(input_file_path, "r") as f:
        video_url_list = yaml.safe_load(f)
    return video_url_list

def video_id_list_loader(video_url_list:dict):
    for video_url in video_url_list['target_video_link']:
        video_id = extract_video_id(video_url)
        video_id_list.append(video_id)
    return video_id_list

def request_to_youtube(video_url_list:dict):
    video_id_list = video_id_list_loader(video_url_list)
    request = youtube.videos().list(
        part = "snippet",
        regionCode = "US",
        id = video_id_list
    )
    return request

def response_from_youtube(video_url_list:dict):
    request = request_to_youtube(video_url_list)
    response = request.execute()['items']
    return response

def extract_info_from_response(response:list[dict]):
    extracted_result = {}
    for item in response:
        extracted_result[item['id']] = {
            'title': item['snippet']['title'],
            'channelTitle': item['snippet']['channelTitle'],
            'description': item['snippet']['description'],
            'publishedAt': item['snippet']['publishedAt'],
            'channelId': item['snippet']['channelId'],
            'defaultLanguage': item['snippet']['defaultLanguage'],
            'localized': item['snippet']['localized'],
            'defaultAudioLanguage': item['snippet']['defaultAudioLanguage'],
            'categoryId': item['snippet']['categoryId'],
            'liveBroadcastContent': item['snippet']['liveBroadcastContent'],
        }
    return extracted_result

def main():
    video_url_list = input_file_loader()
    response = response_from_youtube(video_url_list)
    extracted_result = extract_info_from_response(response)
    return extracted_result

In [57]:
result = main()

In [58]:
result

{'EWvNQjAaOHw': {'title': 'How I use LLMs',
  'channelTitle': 'Andrej Karpathy',
  'description': "The example-driven, practical walkthrough of Large Language Models and their growing list of related features, as a new entry to my general audience series on LLMs. In this more practical followup, I take you through the many ways I use LLMs in my own life.\n\nChapters\n00:00:00 Intro into the growing LLM ecosystem\n00:02:54 ChatGPT interaction under the hood\n00:13:12 Basic LLM interactions examples\n00:18:03 Be aware of the model you're using, pricing tiers\n00:22:54 Thinking models and when to use them\n00:31:00 Tool use: internet search\n00:42:04 Tool use: deep research\n00:50:57 File uploads, adding documents to context\n00:59:00 Tool use: python interpreter, messiness of the ecosystem\n01:04:35 ChatGPT Advanced Data Analysis, figures, plots\n01:09:00 Claude Artifacts, apps, diagrams\n01:14:02 Cursor: Composer, writing code\n01:22:28 Audio (Speech) Input/Output\n01:27:37 Advanced Voi